# Smartphone to DSLR Photo Enhancement using CycleGAN

**Paper:** Zhu et al., *Unpaired Image-to-Image Translation using Cycle-Consistent Adversarial Networks*, ICCV 2017

**Dataset:** iphone2dslr_flower — Smartphone vs DSLR flower photos (unpaired)

---

### Notebook Steps:
1. GPU verification
2. Clone repository
3. Install dependencies
4. Download dataset via Kaggle API
5. Verify dataset
6. Configure training
7. Train CycleGAN
8. Visualize results
9. Test on custom image
10. Download trained weights

## Cell 1 — GPU Verification

In [ ]:
import torch

print('=' * 50)
print('CycleGAN — Environment Check')
print('=' * 50)

if torch.cuda.is_available():
    print(f'GPU  : {torch.cuda.get_device_name(0)}')
    print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print('Status: Ready for training')
else:
    print('No GPU detected!')
    print('Go to: Runtime > Change runtime type > T4 GPU')

print('=' * 50)

## Cell 2 — Clone Repository

In [ ]:
import os

REPO_URL  = 'https://github.com/sidrahrahim5-cmyk/smartphone2dslr-cyclegan.git'
REPO_NAME = 'smartphone2dslr-cyclegan'

if not os.path.exists(REPO_NAME):
    os.system(f'git clone {REPO_URL}')
    print('Repository cloned successfully!')
else:
    os.system(f'cd {REPO_NAME} && git pull')
    print('Repository updated!')

os.chdir(REPO_NAME)
print(f'Working directory: {os.getcwd()}')

## Cell 3 — Install Dependencies

In [ ]:
import os

os.system('pip install -q -r requirements.txt')
print('All dependencies installed!')

## Cell 4 — Download Dataset via Kaggle API

**Before running this cell:**
1. Go to kaggle.com → Profile → Settings → API
2. Click 'Create New Token'
3. kaggle.json will be downloaded to your Downloads folder
4. Upload that file when prompted below

In [ ]:
import os
import zipfile
from google.colab import files

# Step 1: Upload kaggle.json
print('Upload your kaggle.json file...')
uploaded = files.upload()

# Step 2: Configure Kaggle credentials
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
os.system('cp kaggle.json ~/.kaggle/')
os.system('chmod 600 ~/.kaggle/kaggle.json')
print('Kaggle credentials configured!')

# Step 3: Install Kaggle CLI
os.system('pip install -q kaggle')

# Step 4: Download dataset
print('Downloading iphone2dslr flower dataset...')
os.system('kaggle datasets download -d balraj98/iphone2dslr-flower-dataset')

# Step 5: Extract dataset
print('Extracting dataset...')
with zipfile.ZipFile('iphone2dslr-flower-dataset.zip', 'r') as zf:
    zf.extractall('.')

# Step 6: Organize into dataset folders
for split in ['trainA', 'trainB', 'testA', 'testB']:
    os.makedirs(f'dataset/{split}', exist_ok=True)

os.system('cp -r iphone2dslr_flower/trainA/. dataset/trainA/')
os.system('cp -r iphone2dslr_flower/trainB/. dataset/trainB/')
os.system('cp -r iphone2dslr_flower/testA/.  dataset/testA/')
os.system('cp -r iphone2dslr_flower/testB/.  dataset/testB/')

print('\nDataset summary:')
for split in ['trainA', 'trainB', 'testA', 'testB']:
    n = len(os.listdir(f'dataset/{split}'))
    print(f'  dataset/{split}: {n} images')

print('\nDataset ready for training!')

## Cell 5 — Verify Dataset with Sample Images

In [ ]:
import os
import random
import matplotlib.pyplot as plt
from PIL import Image

def show_samples(folder, title, n=4):
    files_list = [
        f for f in os.listdir(folder)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ]
    sample = random.sample(files_list, min(n, len(files_list)))

    fig, axes = plt.subplots(1, n, figsize=(16, 4))
    fig.suptitle(title, fontsize=14, fontweight='bold')

    for ax, fname in zip(axes, sample):
        img = Image.open(os.path.join(folder, fname)).convert('RGB')
        ax.imshow(img)
        ax.axis('off')

    plt.tight_layout()
    plt.show()

show_samples('dataset/trainA', 'Domain A — Smartphone Photos (Input)')
show_samples('dataset/trainB', 'Domain B — DSLR Photos (Target)')

## Cell 6 — Training Configuration

In [ ]:
import sys
sys.path.insert(0, '.')

from config import Config

# Colab-specific overrides
Config.DEVICE        = 'cuda'
Config.EPOCHS        = 50
Config.DECAY_EPOCH   = 25
Config.BATCH_SIZE    = 1
Config.IMG_SIZE      = 256
Config.LOG_EVERY     = 200

Config.make_dirs()

print('Training Configuration')
print('=' * 35)
print(f'Device          : {Config.DEVICE}')
print(f'Epochs          : {Config.EPOCHS}')
print(f'Image size      : {Config.IMG_SIZE}x{Config.IMG_SIZE}')
print(f'Batch size      : {Config.BATCH_SIZE}')
print(f'Learning rate   : {Config.LR}')
print(f'Lambda cycle    : {Config.LAMBDA_CYCLE}')
print(f'Lambda identity : {Config.LAMBDA_IDENTITY}')
print(f'Residual blocks : {Config.N_RESIDUAL_BLOCKS}')
print(f'Replay buffer   : {Config.BUFFER_SIZE}')
print('=' * 35)

## Cell 7 — Start Training

This cell runs the full CycleGAN training loop.
Expected time: ~1.5 hours for 50 epochs on T4 GPU.

In [ ]:
from train import train
train()

## Cell 8 — Visualize Training Progress

In [ ]:
import glob
import matplotlib.pyplot as plt
from PIL import Image

result_files = sorted(glob.glob('assets/results/*.png'))

if not result_files:
    print('No results found. Run training first!')
else:
    indices = [0, len(result_files) // 2, -1]
    labels  = ['Early Training', 'Mid Training', 'Final Result']

    fig, axes = plt.subplots(1, 3, figsize=(20, 7))
    for ax, idx, label in zip(axes, indices, labels):
        img = Image.open(result_files[idx])
        ax.imshow(img)
        ax.set_title(label, fontsize=13, fontweight='bold')
        ax.axis('off')

    plt.suptitle('CycleGAN Training Progress', fontsize=15)
    plt.tight_layout()
    plt.show()
    print(f'Total result files: {len(result_files)}')

## Cell 9 — Test on a Single Image

In [ ]:
import os
import torch
import matplotlib.pyplot as plt
from PIL import Image
import torchvision.transforms as T

from config     import Config
from src.models import Generator
from src.utils  import tensor_to_image

config = Config()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load trained generator
G = Generator(config.IMG_CHANNELS, config.N_RESIDUAL_BLOCKS).to(device)
ckpt = torch.load('checkpoints/latest.pth', map_location=device)
G.load_state_dict(ckpt['G'])
G.eval()
print('Generator loaded successfully!')

# Load a test image from domain A
test_files = os.listdir('dataset/testA')
test_path  = os.path.join('dataset/testA', test_files[0])

transform = T.Compose([
    T.Resize(256),
    T.CenterCrop(256),
    T.ToTensor(),
    T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

pil_img = Image.open(test_path).convert('RGB')
tensor  = transform(pil_img).unsqueeze(0).to(device)

with torch.no_grad():
    enhanced = G(tensor)

# Display side by side comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].imshow(pil_img.resize((256, 256)))
axes[0].set_title('Input: Smartphone Photo', fontsize=13, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(tensor_to_image(enhanced[0]))
axes[1].set_title('Output: DSLR Quality', fontsize=13, fontweight='bold')
axes[1].axis('off')

plt.suptitle('Smartphone to DSLR Enhancement Result', fontsize=15)
plt.tight_layout()
plt.savefig('assets/results/test_result.png', dpi=150, bbox_inches='tight')
plt.show()
print('Test complete! Result saved to assets/results/test_result.png')

## Cell 10 — Download Trained Weights

In [ ]:
import os
from google.colab import files

checkpoint = 'checkpoints/latest.pth'
result     = 'assets/results/test_result.png'

if os.path.exists(checkpoint):
    files.download(checkpoint)
    print('Checkpoint downloaded: latest.pth')
    print('Save it inside your local checkpoints/ folder')
else:
    print('No checkpoint found. Run training first!')

if os.path.exists(result):
    files.download(result)
    print('Test result image downloaded!')